# ViFeedback — Kaggle training

Same contract as `colab_train.ipynb`: **this notebook contains no logic.** It fetches the
repository and calls the same `vifeedback` CLI the laptop runs, so results land in the identical
`results/registry.csv` schema and a platform-only bug is impossible.

## Why Kaggle rather than Colab for this project

| | Kaggle free | Colab free |
|---|---|---|
| GPU | **P100 16 GB** or 2× T4 16 GB | T4 16 GB, not guaranteed |
| Quota | **30 GPU-hours/week, stated** | unstated, throttled by use |
| Session | up to 9 h, rarely preempted | ~12 h, preemptible at any time |
| Output | `/kaggle/working` persisted with the notebook version | lost unless downloaded |

For runs that must finish and be reproducible, the stated quota and the persisted output matter
more than the raw GPU. Both platforms have 16 GB, which is ample: the largest model in this
project's plan, `phobert-large`, needs ~7.9 GB.

## Setup required before running

1. *Settings → Accelerator →* **GPU P100**
2. *Settings → Internet →* **On** (needed to reach the HF Hub)

## What must NOT run here

**Any latency benchmark.** Phase 6 measures CPU p95 on the documented reference machine
(AMD Ryzen 5 6600H, no AVX512-VNNI). A number from a Kaggle VM is not comparable and does not
belong in the registry.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  {p.total_memory/1e9:.2f} GB  capability {p.major}.{p.minor}")
    assert p.total_memory > 10e9, "Enable the GPU accelerator in Settings before running."

## 1. Get the repository

**Route A** clones a remote. **Route B** uses a Kaggle Dataset — upload
`git archive --format=zip -o vifeedback.zip HEAD` as a private dataset and attach it via
*Add Data*. Route B is the better habit on Kaggle: the dataset is versioned and the notebook keeps
working without internet.

In [ ]:
import os
import pathlib
import shutil

REPO_URL = ""  # Route A, e.g. "https://github.com/<user>/ViFeedback-NLP-Service.git"
DATASET_ZIP = "/kaggle/input/vifeedback-repo/vifeedback.zip"  # Route B

WORK = pathlib.Path("/kaggle/working/vifeedback")
if WORK.exists():
    shutil.rmtree(WORK)

if REPO_URL:
    !git clone -q $REPO_URL {WORK}
elif pathlib.Path(DATASET_ZIP).exists():
    WORK.mkdir(parents=True)
    !unzip -qo {DATASET_ZIP} -d {WORK}
else:
    raise SystemExit("Set REPO_URL, or attach the repo zip as a Kaggle Dataset.")

os.chdir(WORK)
print("cwd:", pathlib.Path.cwd())
print(sorted(p.name for p in WORK.iterdir())[:12])

In [ ]:
!pip install -q -e . 2>&1 | tail -3
!pip install -q transformers datasets huggingface-hub typer pyyaml py-cpuinfo 2>&1 | tail -3

# Only if training on a segmented variant. pyvi is the serving choice (ADR-012) and is
# pure Python, so no JVM is needed here either.
NEED_SEGMENTATION = True
if NEED_SEGMENTATION:
    !pip install -q pyvi underthesea 2>&1 | tail -3

import vifeedback

print("vifeedback", vifeedback.__version__)

## 2. Data and integrity

Asserts the official split sizes (11,426 / 1,583 / 3,166) and the leakage figures. If the upstream
data ever shifts, this fails here rather than producing numbers against different data.

In [ ]:
!python -m vifeedback.cli data fetch
!python -m pytest tests/data -q

if NEED_SEGMENTATION:
    !python -m vifeedback.cli data variants --name seg_pyvi

## 3. Train what does not fit on the laptop

The 4.29 GB laptop GPU handles `phobert-base` (3.6 GB measured), `phobert-base-v2` and `visobert`.
Everything below needs the 16 GB here.

`phobert-large` uses batch 16 with `grad_accum` 2, holding the **effective** batch at 32 — the
comparison against `phobert-base` is only meaningful if the effective batch matches.

In [ ]:
# PhoBERT-large — the main reason this notebook exists (~7.9 GB).
!python -m vifeedback.cli train run \
    --task sentiment \
    --model phobert-large \
    --recipe base \
    --preprocessing seg_pyvi \
    --seeds all \
    --epochs 4 \
    --lr 1e-5 \
    --batch-size 16 \
    --max-length 96 \
    --phase 4

In [ ]:
# XLM-R base with unfrozen embeddings — the multilingual control (~5.9 GB).
!python -m vifeedback.cli train run \
    --task sentiment \
    --model xlmr-base \
    --recipe base \
    --preprocessing seg_pyvi \
    --seeds all \
    --epochs 4 \
    --batch-size 32 \
    --phase 4

## 4. Bring the results home

`/kaggle/working` is saved with the notebook version, so *Save Version → Output* already persists
everything. The zip below is for downloading it in one file.

Merge **by `run_id`**, never by blind append: the archive's `registry.csv` may contain rows that
were already committed before the clone.

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/kaggle_results", "zip", "results")
print("written: /kaggle/working/kaggle_results.zip")

# On the laptop:
#   unzip -o kaggle_results.zip -d /tmp/kag
#   cp -rn /tmp/kag/runs/* results/runs/
#   python - <<'EOF'
#   import pandas as pd
#   a = pd.read_csv('results/registry.csv'); b = pd.read_csv('/tmp/kag/registry.csv')
#   out = pd.concat([a, b]).drop_duplicates(subset='run_id', keep='first')
#   out.to_csv('results/registry.csv', index=False)
#   print(f'{len(out) - len(a)} new rows merged')
#   EOF

## 5. Reporting the split honestly

Rows produced here carry a different `env.json` — P100 rather than RTX 3050, different driver,
possibly different library versions. That is irrelevant for **accuracy**, which is
hardware-independent, and disqualifying for **latency**, which is not.

Mark Kaggle-trained rows in the final results table and name the GPU. A reviewer should never have
to guess which machine produced a number.